# LLaMA 3.2 Privacy QA Classification Analysis

This notebook analyzes the performance of LLaMA 3.2-3B-Instruct model on the Privacy QA classification task.

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import numpy as np

## Load and Prepare Data

In [ ]:
# === Load the model output CSV ===
OUTFILE = "../test_results/llama_3b_privacy_qa.csv"

# Load results
df = pd.read_csv(OUTFILE)
print(f"Loaded {len(df)} records")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Display first few rows
df.head()

## Data Preprocessing

In [ ]:
# === Normalize and map labels ===
def normalize_label(label):
    if not isinstance(label, str):
        return None
    label = label.strip().lower()
    if "relevant" in label:
        # specifically handle 'irrelevant' before 'relevant'
        if "irrelevant" in label:
            return 0
        return 1
    return None

df["gold"] = df["Any_Relevant"].apply(lambda x: 1 if str(x).strip().lower() == "relevant" else 0)
df["pred"] = df["final_label"].apply(normalize_label)

print(f"Gold label distribution:")
print(df["gold"].value_counts())
print(f"\nPredicted label distribution (before dropping NaN):")
print(df["pred"].value_counts(dropna=False))

In [ ]:
# === Drop rows with missing predictions ===
df_clean = df.dropna(subset=["pred"])
print(f"Records after dropping NaN predictions: {len(df_clean)}")
print(f"Dropped {len(df) - len(df_clean)} records with missing predictions")

## Model Performance Evaluation

In [ ]:
# === Compute metrics ===
acc = accuracy_score(df_clean["gold"], df_clean["pred"])
f1 = f1_score(df_clean["gold"], df_clean["pred"])
cm = confusion_matrix(df_clean["gold"], df_clean["pred"])
report = classification_report(df_clean["gold"], df_clean["pred"], target_names=["Irrelevant", "Relevant"])

# === Print results ===
print("===============================================")
print("🔍  LLaMA 3.2 Privacy QA Evaluation")
print("===============================================")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nConfusion Matrix (Counts):")
print(f" {cm}")
print("\nConfusion Matrix (Normalized):")
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, None]
print(f" {cm_normalized}")
print("\nDetailed Classification Report:")
print(report)

## Confusion Matrix Visualization

In [ ]:
# === Plot confusion matrix ===
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix — LLaMA 3.2 Privacy QA", fontsize=14, fontweight='bold')
plt.colorbar()

classes = ["Irrelevant", "Relevant"]
tick_marks = range(len(classes))
plt.xticks(tick_marks, classes, rotation=45)
plt.yticks(tick_marks, classes)

# Label cells with counts
thresh = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j, i, format(cm[i, j], "d"),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
            fontsize=12, fontweight='bold'
        )

plt.tight_layout()
plt.ylabel("True Label", fontsize=12)
plt.xlabel("Predicted Label", fontsize=12)
plt.show()

## Performance Metrics Breakdown

In [ ]:
# Calculate individual metrics
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix Breakdown:")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")

print("\nCalculated Metrics:")
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"Precision (Relevant): {precision:.4f}")
print(f"Recall (Relevant): {recall:.4f}")
print(f"Specificity (Irrelevant): {specificity:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Accuracy: {acc:.4f}")

## Error Analysis

In [ ]:
# Analyze prediction errors
df_clean['correct'] = df_clean['gold'] == df_clean['pred']

print("Prediction Accuracy by Class:")
print(df_clean.groupby('gold')['correct'].agg(['count', 'sum', 'mean']))

print("\nError Types:")
false_positives = df_clean[(df_clean['gold'] == 0) & (df_clean['pred'] == 1)]
false_negatives = df_clean[(df_clean['gold'] == 1) & (df_clean['pred'] == 0)]

print(f"False Positives (Irrelevant → Relevant): {len(false_positives)}")
print(f"False Negatives (Relevant → Irrelevant): {len(false_negatives)}")

## Sample Predictions

In [ ]:
# Show some example predictions
print("=== CORRECT PREDICTIONS ===")
print("\nCorrect Relevant Predictions:")
correct_relevant = df_clean[(df_clean['gold'] == 1) & (df_clean['pred'] == 1)]
if len(correct_relevant) > 0:
    sample = correct_relevant.sample(min(3, len(correct_relevant)))
    for idx, row in sample.iterrows():
        print(f"\nQuery: {row['Query'][:100]}...")
        print(f"Segment: {row['Segment'][:100]}...")
        print(f"Reasoning: {row['reasoning']}")

print("\n=== INCORRECT PREDICTIONS ===")
print("\nFalse Positives (Irrelevant classified as Relevant):")
if len(false_positives) > 0:
    sample = false_positives.sample(min(3, len(false_positives)))
    for idx, row in sample.iterrows():
        print(f"\nQuery: {row['Query'][:100]}...")
        print(f"Segment: {row['Segment'][:100]}...")
        print(f"Reasoning: {row['reasoning']}")

## Summary

This analysis shows the performance of LLaMA 3.2-3B-Instruct on the Privacy QA classification task. Key findings:

- **Overall Performance**: The model achieves moderate accuracy but struggles with precision
- **Class Imbalance**: The model tends to over-classify segments as relevant
- **Trade-offs**: Higher recall for relevant class but at the cost of precision

The results can be compared with other models (like Qwen 2.5) to understand relative performance on this privacy document classification task.